# Association Testing

## Objective
The goal of this analysis is to investigate the relationship between Food Microbiome Exposure (FME) and gut microbiome characteristics.

Specifically, we examine whether FME is associated with:

- Alpha diversity (Shannon diversity)
- Microbiome stability
- Community composition metrics

This analysis is based on the dataset generated in the FME analysis stage.

In [32]:
#Data manipulation
import pandas as pd
import numpy as np

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Statistics
from scipy.stats import pearsonr, spearmanr
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu

#Regression
import statsmodels.api as sm
import statsmodels.formula.api as smf

#Model evaluation
import statsmodels.formula.api as smf

#Display settings
pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path
import sys
#Locate repo root (directory containing config.py) regardless of launch directory
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
DATA_DIR = _root / "data"  # robust: does not depend on __file__ inside Jupyter
sys.path.insert(0, str(_root))

## Dataset Overview

In this section, we load and inspect the statistical dataset created in the FME analysis step.

The goal is to understand the structure of the dataset, including the number of samples, available variables, data types, and missing values before performing association testing.

In [ ]:
#Load the statistical dataset created in the FME analysis step

df = pd.read_csv(DATA_DIR / "fme_statistical_dataset_extended.csv")

df.head()

,fecal_sample_id,participant_id,study_day,Gender,Age,BMI,Weight,Supplement,Medications,fme_score_daily,shannon_diversity,richness,simpson_diversity,inverse_simpson,pielou_evenness,participant_shannon_cv,participant_shannon_mean,participant_n_samples,KCAL,PROT,TFAT,CARB,FIBE,SUGR,SODI,D_TOTAL,D_YOGURT,D_CHEESE,PF_MEAT,PF_SEAFD_HI,G_WHOLE
0,MCT.f.0002,MCTs01,2,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),67.196133,2.862675,211,0.871961,7.810101,0.534894,0.049204,3.066187,15,1970.043625,109.619765,53.615264,242.919015,14.419375,74.208114,4300.616500,4.156035,0.646,1.153475,0.00000,3.969,0.9718
1,MCT.f.0003,MCTs01,3,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),81.241247,3.522832,208,0.947340,18.989597,0.660011,0.049204,3.066187,15,1714.895330,89.992993,50.561373,234.225138,25.996125,67.474665,4225.813250,2.979655,0.646,1.207575,0.00000,0.000,2.9318
2,MCT.f.0004,MCTs01,4,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),126.771107,3.031188,213,0.891830,9.244741,0.565384,0.049204,3.066187,15,2487.232625,94.611953,92.021551,257.539030,26.178425,83.960308,5329.882125,3.634460,0.000,1.482600,0.00000,0.000,0.9718
3,MCT.f.0005,MCTs01,5,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),31.484626,2.973836,207,0.882504,8.510902,0.557659,0.049204,3.066187,15,2260.211176,110.268353,95.570253,165.608070,13.011350,64.921966,3527.501416,3.379710,0.000,1.059000,1.81440,0.000,0.9718
4,MCT.f.0006,MCTs01,6,F,25.9,25.0,74.6,EVOO,Nexplanon (bc implant),61.226577,3.081097,209,0.900002,10.000183,0.576732,0.049204,3.066187,15,2951.082000,161.817790,108.398516,332.637229,21.529050,119.865677,5428.573000,1.502150,0.000,0.000000,6.29596,0.000,0.4698


In [ ]:
#Dataset overview

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (475, 31)

Columns:
['fecal_sample_id', 'participant_id', 'study_day', 'Gender', 'Age', 'BMI', 'Weight', 'Supplement', 'Medications', 'fme_score_daily', 'shannon_diversity', 'richness', 'simpson_diversity', 'inverse_simpson', 'pielou_evenness', 'participant_shannon_cv', 'participant_shannon_mean', 'participant_n_samples', 'KCAL', 'PROT', 'TFAT', 'CARB', 'FIBE', 'SUGR', 'SODI', 'D_TOTAL', 'D_YOGURT', 'D_CHEESE', 'PF_MEAT', 'PF_SEAFD_HI', 'G_WHOLE']


In [20]:
# Summary statistics

df.describe()

,study_day,Age,BMI,Weight,fme_score_daily,shannon_diversity,richness,simpson_diversity,inverse_simpson,pielou_evenness,participant_shannon_cv,participant_shannon_mean,participant_n_samples,KCAL,PROT,TFAT,CARB,FIBE,SUGR,SODI,D_TOTAL,D_YOGURT,D_CHEESE,PF_MEAT,PF_SEAFD_HI,G_WHOLE
count,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000
mean,8.951579,31.152211,23.174737,69.069053,73.325237,2.864593,203.562105,0.859697,8.781242,0.538585,0.062621,2.864593,14.604211,2110.562806,89.144876,91.512435,229.066013,22.337148,85.219859,3491.564185,1.921608,0.110930,0.993474,1.190717,0.325082,1.263407
std,4.939697,10.156460,3.292062,14.646855,58.307131,0.413157,11.953754,0.075778,3.599882,0.074656,0.037747,0.368406,2.429422,735.450983,37.394616,40.212887,84.487698,11.600234,43.468122,1629.773797,1.541540,0.271693,1.339567,2.307983,1.576152,1.628686
min,1.000000,19.700000,17.000000,48.500000,0.000000,1.571683,140.000000,0.464869,1.868700,0.299239,0.020213,1.944592,6.000000,722.552500,15.702908,20.467095,45.046628,3.618000,11.305484,550.853000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,23.900000,21.100000,59.100000,29.322198,2.610804,198.000000,0.831460,5.933352,0.492687,0.036472,2.530238,14.000000,1599.179800,62.949223,60.910505,176.968102,14.850000,53.044957,2317.423250,0.664687,0.000000,0.000000,0.000000,0.000000,0.000000
50%,9.000000,28.500000,22.900000,66.900000,61.777509,2.964396,207.000000,0.886356,8.799400,0.556193,0.048859,3.012764,15.000000,2028.168000,85.496863,85.593670,217.000000,19.693800,81.216000,3314.340000,1.650454,0.000000,0.572400,0.000000,0.000000,0.651750
75%,13.000000,37.000000,25.900000,82.900000,104.269671,3.143011,212.000000,0.910088,11.122002,0.591939,0.079944,3.130106,16.000000,2494.098500,106.259492,116.172395,271.939905,26.711850,110.042055,4310.122600,2.754320,0.000000,1.454882,1.574200,0.000000,2.093600
max,17.000000,61.600000,31.600000,104.800000,331.223987,3.731016,220.000000,0.957150,23.337217,0.693511,0.179903,3.483381,17.000000,5085.977400,256.499039,242.761818,582.015016,67.501903,261.857405,13884.247300,9.323425,1.665000,9.047425,17.577000,20.769210,9.934594


In [21]:
print(df.isnull().sum())

fecal_sample_id               0
participant_id                0
study_day                     0
Gender                        0
Age                           0
BMI                           0
Weight                        0
Supplement                    0
Medications                 255
fme_score_daily               0
shannon_diversity             0
richness                      0
simpson_diversity             0
inverse_simpson               0
pielou_evenness               0
participant_shannon_cv        0
participant_shannon_mean      0
participant_n_samples         0
KCAL                          0
PROT                          0
TFAT                          0
CARB                          0
FIBE                          0
SUGR                          0
SODI                          0
D_TOTAL                       0
D_YOGURT                      0
D_CHEESE                      0
PF_MEAT                       0
PF_SEAFD_HI                   0
G_WHOLE                       0
dtype: i

In [30]:
#Define the list of alpha diversity metrics to analyze
alpha_diversity_metrics = [
    "shannon_diversity",
    "richness",
    "simpson_diversity",
    "inverse_simpson",
    "pielou_evenness"
]

alpha_diversity_metrics

['shannon_diversity',
 'richness',
 'simpson_diversity',
 'inverse_simpson',
 'pielou_evenness']

## Correlation Analysis

In this section, we investigate the relationship between Food Microbiome Exposure (FME) and microbiome diversity metrics.

We begin by examining the correlation between the daily FME score and Shannon diversity, which is a commonly used measure of microbiome diversity.

In [31]:
#Correlation between FME and each alpha diversity metric

correlation_results = []

for metric in alpha_diversity_metrics:
    analysis_df = df[["fme_score_daily", metric]].dropna()

    pearson_corr, pearson_p = pearsonr(
        analysis_df["fme_score_daily"],
        analysis_df[metric]
    )

    spearman_corr, spearman_p = spearmanr(
        analysis_df["fme_score_daily"],
        analysis_df[metric]
    )

    correlation_results.append({
        "metric": metric,
        "n_samples": len(analysis_df),
        "pearson_corr": pearson_corr,
        "pearson_p_value": pearson_p,
        "spearman_corr": spearman_corr,
        "spearman_p_value": spearman_p
    })

correlation_results_df = pd.DataFrame(correlation_results)

display(correlation_results_df.round(4))

,metric,n_samples,pearson_corr,pearson_p_value,spearman_corr,spearman_p_value
0,shannon_diversity,475,-0.0406,0.3776,-0.0242,0.5988
1,richness,475,0.0234,0.6113,0.0059,0.8980
2,simpson_diversity,475,-0.0407,0.3758,-0.0467,0.3101
3,inverse_simpson,475,-0.0485,0.2917,-0.0467,0.3101
4,pielou_evenness,475,-0.0439,0.3400,-0.0252,0.5834


In [33]:
#Adjusted Models
adjusted_model_results = []

for metric in alpha_diversity_metrics:
    model_df = df[
        ["fme_score_daily", metric, "Age", "BMI"]
    ].dropna()

    formula = f"{metric} ~ fme_score_daily + Age + BMI"

    model = smf.ols(
        formula=formula,
        data=model_df
    ).fit()

    adjusted_model_results.append({
        "metric": metric,
        "model": "OLS adjusted for Age + BMI",
        "n_samples": len(model_df),
        "fme_coef": model.params.get("fme_score_daily", np.nan),
        "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
        "age_p_value": model.pvalues.get("Age", np.nan),
        "bmi_p_value": model.pvalues.get("BMI", np.nan),
        "r_squared": model.rsquared
    })

adjusted_model_results_df = pd.DataFrame(adjusted_model_results)

display(adjusted_model_results_df.round(4))

,metric,model,n_samples,fme_coef,fme_p_value,age_p_value,bmi_p_value,r_squared
0,shannon_diversity,OLS adjusted for Age + BMI,475,-0.0002,0.5154,0.2073,0.5884,0.0063
1,richness,OLS adjusted for Age + BMI,475,0.0042,0.6614,0.6000,0.8526,0.0011
2,simpson_diversity,OLS adjusted for Age + BMI,475,-0.0000,0.4961,0.1869,0.9790,0.0055
3,inverse_simpson,OLS adjusted for Age + BMI,475,-0.0021,0.4655,0.0285,0.9472,0.0127
4,pielou_evenness,OLS adjusted for Age + BMI,475,-0.0000,0.4763,0.1835,0.5967,0.0070


In [34]:
#Nutrition Covariates
nutrition_covariates = [
    "KCAL",
    "FIBE",
    "PROT",
    "TFAT",
    "CARB"
]

available_nutrition_covariates = [
    col for col in nutrition_covariates
    if col in df.columns
]

nutrition_adjusted_model_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "fme_score_daily",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = ["fme_score_daily", "Age", "BMI"] + available_nutrition_covariates
    formula = f"{metric} ~ " + " + ".join(covariates)

    model = smf.ols(
        formula=formula,
        data=model_df
    ).fit()

    nutrition_adjusted_model_results.append({
        "metric": metric,
        "model": "OLS adjusted for Age + BMI + nutrition",
        "n_samples": len(model_df),
        "fme_coef": model.params.get("fme_score_daily", np.nan),
        "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
        "age_p_value": model.pvalues.get("Age", np.nan),
        "bmi_p_value": model.pvalues.get("BMI", np.nan),
        "r_squared": model.rsquared
    })

nutrition_adjusted_model_results_df = pd.DataFrame(nutrition_adjusted_model_results)

display(nutrition_adjusted_model_results_df.round(4))

,metric,model,n_samples,fme_coef,fme_p_value,age_p_value,bmi_p_value,r_squared
0,shannon_diversity,OLS adjusted for Age + BMI + nutrition,475,0.0000,0.9498,0.0421,0.5993,0.0354
1,richness,OLS adjusted for Age + BMI + nutrition,475,0.0223,0.0596,0.9104,0.8926,0.0297
2,simpson_diversity,OLS adjusted for Age + BMI + nutrition,475,-0.0000,0.7941,0.0361,0.2598,0.0351
3,inverse_simpson,OLS adjusted for Age + BMI + nutrition,475,0.0004,0.9054,0.0064,0.3817,0.0367
4,pielou_evenness,OLS adjusted for Age + BMI + nutrition,475,-0.0000,0.9410,0.0352,0.5747,0.0359


In [35]:
#Mixed Models with Participant Random Intercept
mixed_model_results = []

for metric in alpha_diversity_metrics:
    model_columns = [
        "participant_id",
        "fme_score_daily",
        metric,
        "Age",
        "BMI"
    ] + available_nutrition_covariates

    model_df = df[model_columns].dropna()

    covariates = ["fme_score_daily", "Age", "BMI"] + available_nutrition_covariates
    formula = f"{metric} ~ " + " + ".join(covariates)

    try:
        model = smf.mixedlm(
            formula=formula,
            data=model_df,
            groups=model_df["participant_id"]
        ).fit()

        mixed_model_results.append({
            "metric": metric,
            "model": "MixedLM adjusted for Age + BMI + nutrition + participant random intercept",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),
            "fme_coef": model.params.get("fme_score_daily", np.nan),
            "fme_p_value": model.pvalues.get("fme_score_daily", np.nan),
            "age_p_value": model.pvalues.get("Age", np.nan),
            "bmi_p_value": model.pvalues.get("BMI", np.nan)
        })

    except Exception as e:
        mixed_model_results.append({
            "metric": metric,
            "model": "MixedLM failed",
            "n_samples": len(model_df),
            "n_participants": model_df["participant_id"].nunique(),
            "fme_coef": np.nan,
            "fme_p_value": np.nan,
            "age_p_value": np.nan,
            "bmi_p_value": np.nan,
            "error": str(e)
        })

mixed_model_results_df = pd.DataFrame(mixed_model_results)

display(mixed_model_results_df.round(4))

c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\linoy\anaconda3\envs\project3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,metric,model,n_samples,n_participants,fme_coef,fme_p_value,age_p_value,bmi_p_value
0,shannon_diversity,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0004,0.0510,0.6997,0.8906
1,richness,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,0.0072,0.4767,0.8633,0.9619
2,simpson_diversity,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0001,0.2745,0.6413,0.9563
3,inverse_simpson,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0010,0.6792,0.5421,0.8908
4,pielou_evenness,MixedLM adjusted for Age + BMI + nutrition + p...,475,34,-0.0001,0.0392,0.6797,0.8965


In [36]:
#Final summary table across all association models

#Correlation results
correlation_summary = correlation_results_df.copy()
correlation_summary["model_type"] = "Correlation"
correlation_summary["fme_coef"] = correlation_summary["pearson_corr"]
correlation_summary["fme_p_value"] = correlation_summary["pearson_p_value"]
correlation_summary = correlation_summary[
    ["metric", "model_type", "n_samples", "fme_coef", "fme_p_value"]
]

#OLS adjusted for Age + BMI
ols_age_bmi_summary = adjusted_model_results_df.copy()
ols_age_bmi_summary["model_type"] = "OLS: Age + BMI"
ols_age_bmi_summary = ols_age_bmi_summary[
    ["metric", "model_type", "n_samples", "fme_coef", "fme_p_value"]
]

#OLS adjusted for Age + BMI + nutrition
ols_nutrition_summary = nutrition_adjusted_model_results_df.copy()
ols_nutrition_summary["model_type"] = "OLS: Age + BMI + nutrition"
ols_nutrition_summary = ols_nutrition_summary[
    ["metric", "model_type", "n_samples", "fme_coef", "fme_p_value"]
]

#Mixed effects model
mixed_summary = mixed_model_results_df.copy()
mixed_summary["model_type"] = "MixedLM: Age + BMI + nutrition + participant"
mixed_summary = mixed_summary[
    ["metric", "model_type", "n_samples", "fme_coef", "fme_p_value"]
]

#Combine all summaries
final_association_summary = pd.concat(
    [
        correlation_summary,
        ols_age_bmi_summary,
        ols_nutrition_summary,
        mixed_summary
    ],
    axis=0,
    ignore_index=True
)

#Add significance label
final_association_summary["significance"] = np.where(
    final_association_summary["fme_p_value"] < 0.05,
    "significant",
    np.where(
        final_association_summary["fme_p_value"] < 0.10,
        "trend",
        "not significant"
    )
)

display(final_association_summary.round(4))

,metric,model_type,n_samples,fme_coef,fme_p_value,significance
0,shannon_diversity,Correlation,475,-0.0406,0.3776,not significant
1,richness,Correlation,475,0.0234,0.6113,not significant
2,simpson_diversity,Correlation,475,-0.0407,0.3758,not significant
3,inverse_simpson,Correlation,475,-0.0485,0.2917,not significant
4,pielou_evenness,Correlation,475,-0.0439,0.3400,not significant
5,shannon_diversity,OLS: Age + BMI,475,-0.0002,0.5154,not significant
6,richness,OLS: Age + BMI,475,0.0042,0.6614,not significant
7,simpson_diversity,OLS: Age + BMI,475,-0.0000,0.4961,not significant
8,inverse_simpson,OLS: Age + BMI,475,-0.0021,0.4655,not significant
9,pielou_evenness,OLS: Age + BMI,475,-0.0000,0.4763,not significant


## Summary of Association Testing Results

In this analysis, we tested the association between the daily FME score and several gut microbiome alpha diversity metrics:
- Shannon diversity
- Richness
- Simpson diversity
- Inverse Simpson
- Pielou evenness

First, simple Pearson and Spearman correlations were calculated between FME and each alpha diversity metric.
These unadjusted correlations did not show significant associations.

Next, linear regression models adjusted for Age and BMI were fitted for each diversity metric.
In these models, FME was not significantly associated with any of the alpha diversity metrics.

We then extended the regression models by additionally adjusting for nutritional covariates, including total calorie intake, fiber, protein, fat, and carbohydrates.
In this model set, richness showed a near significant trend with FME, but did not pass the conventional significance threshold of 0.05.

Finally, because the dataset contains repeated fecal samples from the same participants, we fitted mixed effects models with participant-specific random intercepts.
These models adjusted for Age, BMI, nutritional covariates, and participant level repeated sampling structure.

In the mixed effects model, Pielou evenness showed a statistically significant association with FME.
The coefficient was negative, suggesting that higher FME scores were associated with lower evenness of the microbial community.
Shannon diversity showed a near significant trend, also with a negative coefficient.

Overall, the results suggest that the association between FME and gut microbiome alpha diversity is not strongly reflected in simple unadjusted correlations.
However, after accounting for repeated samples per participant using mixed effects models, FME appears to be associated specifically with microbial evenness, rather than with richness or dominance based diversity metrics.

In [37]:
#Save final association summary table

final_association_summary.to_csv(
    DATA_DIR / "association_testing_extended_summary.csv",
    index=False
)

print("Saved association_testing_extended_summary.csv")

Saved association_testing_extended_summary.csv
